# Datathon 2026 — Phần 3: Sales Forecasting Pipeline
> **Tác giả:** Pờ tít Hê nô | **Thư viện:** LightGBM · Prophet · Ridge · lunarcalendar


# Datathon 2026 — Sales Forecasting Pipeline
### 📋 Tóm tắt Phương pháp luận & Tuân thủ Ràng buộc
1. **Kiểm soát rò rỉ dữ liệu (No-Leakage):** Tuyệt đối không sử dụng biến mục tiêu từ tập test. Toàn bộ đặc trưng được xây dựng từ chuỗi thời gian và các giá trị trễ lịch sử.
2. **Validation theo chiều thời gian:** Sử dụng chiến lược Time-series Hold-out (2022-07-04) để đánh giá mô hình.
3. **Khả năng giải thích (Explainability):** Mô hình được giải thích chi tiết bằng giá trị SHAP.
4. **Không sử dụng dữ liệu ngoài:** Chỉ sử dụng bộ dữ liệu chính thức được cung cấp.

## 0. Cài đặt thư viện


In [ ]:
# !pip install lightgbm prophet lunarcalendar --quiet

import logging
import warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

In [ ]:
from pathlib import Path

def resolve_data_dir():
    for p in [
        Path('/kaggle/input/competitions/datathon-2026-round-1'),
        Path('/kaggle/input/datathon-2026-round-1'),
        Path('./dataset'), Path('.'),
    ]:
        if (p / 'sales.csv').exists() and (p / 'sample_submission.csv').exists():
            return p
    raise FileNotFoundError('Không tìm thấy file sales.csv / sample_submission.csv')

DATA_DIR = resolve_data_dir()
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'DATA_DIR: {DATA_DIR}')
print(f'OUT_DIR : {OUT_DIR}')


## 1. Tải dữ liệu


In [ ]:
sales = pd.read_csv(DATA_DIR / 'sales.csv', parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

sales['Y'] = sales['Date'].dt.year
sales['Q'] = sales['Date'].dt.quarter
sales['M'] = sales['Date'].dt.month
sales['DOW'] = sales['Date'].dt.dayofweek
sales['day'] = sales['Date'].dt.day

print(sales.shape, sales['Date'].min().date(), '->', sales['Date'].max().date())
sales.head()

## 2. Khám phá dữ liệu (EDA)


### 2.1 Xu hướng Doanh thu & COGS theo thời gian


In [ ]:
# EDA: Xem xét dòng thời gian
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(sales.Date, sales.Revenue/1e6, lw=0.5)
axes[0].set_ylabel('Revenue (M)'); axes[0].set_title('Revenue 2012-2022')
axes[1].plot(sales.Date, sales.COGS/1e6, lw=0.5, color='red')
axes[1].set_ylabel('COGS (M)')
plt.tight_layout(); plt.show()

### 2.2 Tính mùa vụ theo tháng


In [ ]:
# EDA: Tính mùa vụ theo tháng
monthly = sales.groupby(['Y','M']).Revenue.mean().reset_index()
fig, ax = plt.subplots(figsize=(12, 5))
for y in sorted(sales.Y.unique()):
    d = monthly[monthly.Y == y]
    ax.plot(d.M, d.Revenue/1e6, marker='o', label=str(y))
ax.set_xticks(range(1,13)); ax.legend(ncol=4, fontsize=8)
ax.set_title('Mua vu thang qua cac nam')
plt.show()

### 2.3 Biên lợi nhuận theo quý (chẵn/lẻ)


In [ ]:
# EDA: Biên lợi nhuận theo quý
q_marg = sales.groupby(['Y','Q']).apply(
    lambda d: d.COGS.sum()/d.Revenue.sum(), include_groups=False).reset_index()
q_marg.columns = ['Y','Q','margin']

fig, ax = plt.subplots(figsize=(11, 4))
for q in [1,2,3,4]:
    d = q_marg[q_marg.Q == q]
    ax.plot(d.Y, d.margin, marker='o', label=f'Q{q}')
ax.axhline(0.9, color='gray', ls='--')
ax.set_title('Margin theo quy - Q3 zigzag chan/le')
ax.legend(); plt.show()

## 3. Kỹ thuật Đặc trưng (Feature Engineering)


In [ ]:
# ---- Cấu hình LỊCH KHUYẾN MÃI Chuẩn (Theo notebook0ed0a2457f) ----
PROMO_SCHEDULE = [
    ('spring_sale',   3, 18, 30, 12, True),
    ('mid_year',      6, 23, 29, 18, True),
    ('fall_launch',   8, 30, 32, 10, True),
    ('year_end',     11, 18, 45, 20, True),
    ('urban_blowout', 7, 30, 34, 50,  'odd'),  # 30/07 -> 02/09 (inclusive)
    ('rural_special', 1, 30, 30, 15,  'odd'),
]

TET_DATES = {
    2013:'2013-02-10', 2014:'2014-01-31', 2015:'2015-02-19',
    2016:'2016-02-08', 2017:'2017-01-28', 2018:'2018-02-16',
    2019:'2019-02-05', 2020:'2020-01-25', 2021:'2021-02-12',
    2022:'2022-02-01', 2023:'2023-01-22', 2024:'2024-02-10',
}

VN_FIXED_HOLIDAYS = [
    (1,1,'new_year'), (3,8,'womens_day'), (4,30,'reunification'),
    (5,1,'labor_day'), (9,2,'national_day'), (10,20,'vn_womens_day'),
    (11,11,'dd_1111'), (12,12,'dd_1212'),
    (12,24,'christmas_eve'), (12,25,'christmas'),
]


In [ ]:
# ---- Feature Engineering (Simplified to match notebook0ed0a2457f) ----
def build_features(dates):
    df = pd.DataFrame({'Date': pd.to_datetime(dates)})
    d = df['Date']

    # Calendar
    df['year']    = d.dt.year
    df['month']   = d.dt.month
    df['day']     = d.dt.day
    df['dow']     = d.dt.dayofweek
    df['doy']     = d.dt.dayofyear
    df['quarter'] = d.dt.quarter
    df['is_weekend']    = (df['dow']>=5).astype(int)
    df['days_to_eom']   = d.dt.days_in_month - df['day']
    df['days_from_som'] = df['day'] - 1
    df['dim']           = d.dt.days_in_month

    # Edge of month
    for k in [1,2,3]:
        df[f'is_last{k}']  = (df['days_to_eom']  <= k-1).astype(int)
        df[f'is_first{k}'] = (df['days_from_som'] <= k-1).astype(int)

    # Trend + regime
    df['t_days']  = (d - pd.Timestamp('2020-01-01')).dt.days
    df['t_years'] = df['t_days']/365.25
    df['regime_pre2019']  = (df['year']<=2018).astype(int)
    df['regime_2019']     = (df['year']==2019).astype(int)
    df['regime_post2019'] = (df['year']>=2020).astype(int)

    # Fourier
    TAU = 2*np.pi
    for k in (1,2,3,4,5):
        df[f'sin_y{k}'] = np.sin(TAU*k*df['doy']/365.25)
        df[f'cos_y{k}'] = np.cos(TAU*k*df['doy']/365.25)
    for k in (1,2):
        df[f'sin_w{k}'] = np.sin(TAU*k*df['dow']/7.0)
        df[f'cos_w{k}'] = np.cos(TAU*k*df['dow']/7.0)
    for k in (1,2):
        df[f'sin_m{k}'] = np.sin(TAU*k*(df['day']-1)/df['dim'])
        df[f'cos_m{k}'] = np.cos(TAU*k*(df['day']-1)/df['dim'])

    # Holidays
    for (m, dd_, name) in VN_FIXED_HOLIDAYS:
        df[f'hol_{name}'] = ((df['month']==m) & (df['day']==dd_)).astype(int)

    # Tet distance
    tet_lut = {y: pd.Timestamp(v) for y,v in TET_DATES.items()}
    def nearest_tet_diff(dd):
        cands = [tet_lut.get(dd.year), tet_lut.get(dd.year-1), tet_lut.get(dd.year+1)]
        cands = [c for c in cands if c is not None]
        valid = [(dd-c).days for c in cands if abs((dd-c).days)<=45]
        return min(valid) if valid else 999
    diffs = np.array([nearest_tet_diff(dd) for dd in d])
    df['tet_days_diff'] = diffs
    df['tet_in_7']      = (np.abs(diffs)<=7).astype(int)
    df['tet_in_14']     = (np.abs(diffs)<=14).astype(int)
    df['tet_before_7']  = ((diffs>=-7) & (diffs<0)).astype(int)
    df['tet_after_7']   = ((diffs>0) & (diffs<=7)).astype(int)
    df['tet_on']        = (diffs==0).astype(int)

    # Black Friday
    def is_bf(dd):
        if dd.month != 11: return 0
        last = pd.Timestamp(year=dd.year, month=11, day=30)
        last_fri = last - pd.Timedelta(days=(last.dayofweek - 4) % 7)
        return int(dd == last_fri)
    df['hol_black_friday'] = [is_bf(dd) for dd in d]

    # Promo windows
    yrs = sorted(set(df['year'].tolist()))
    for (name, sm, sd, dur, disc, recur) in PROMO_SCHEDULE:
        in_prom = np.zeros(len(df), dtype=int)
        since   = np.full(len(df), -1.0)
        until   = np.full(len(df), -1.0)
        discount= np.zeros(len(df))
        for y in range(min(yrs)-1, max(yrs)+2):
            if recur=='odd' and y%2==0: continue
            start = pd.Timestamp(year=y, month=sm, day=sd)
            end   = start + pd.Timedelta(days=dur)
            mask  = (d>=start) & (d<=end)
            in_prom[mask] = 1
            since[mask]   = (d[mask]-start).dt.days
            until[mask]   = (end-d[mask]).dt.days
            discount[mask]= disc or 0
        df[f'promo_{name}']       = in_prom
        df[f'promo_{name}_since'] = since
        df[f'promo_{name}_until'] = until
        df[f'promo_{name}_disc']  = discount

    df['is_odd_year'] = (df['year'] % 2).astype(int)
    return df

print('Feature check:', build_features(pd.date_range('2023-01-01', '2023-01-05')).shape)


## 4. Chuẩn bị tập Train & Test


In [ ]:
feat = build_features(sales['Date'])
feat['Revenue'] = sales['Revenue'].values
feat['COGS']    = sales['COGS'].values

test_dates = pd.date_range('2023-01-01', '2024-07-01', freq='D')
test_df = build_features(test_dates)

# ---- Đặc trưng quá khứ (thống kê lịch sử + dịch chuyển thời gian về năm trước) ----
# Giúp LightGBM bắt được chuỗi giá trị trong quá khứ thay vì chỉ dựa vào lịch.
def build_lag_features(dates, sales_):
    s = sales_.sort_values('Date').reset_index(drop=True).copy()
    s['Y'] = s['Date'].dt.year
    s['M'] = s['Date'].dt.month
    s['D'] = s['Date'].dt.day
    s['DOW'] = s['Date'].dt.dayofweek
    s['WOY'] = s['Date'].dt.isocalendar().week.astype(int)
    # Trọng số theo năm: năm càng gần thì trọng số càng cao
    yr_min = int(s['Y'].min())
    s['_w'] = (s['Y'] - yr_min + 1).astype(float)

    def wavg(d, col):
        return float(np.average(d[col], weights=d['_w']))

    md = s.groupby(['M','D']).apply(lambda d: pd.Series({
        'rev': wavg(d,'Revenue'), 'cog': wavg(d,'COGS')}))
    dw = s.groupby(['DOW','WOY']).apply(lambda d: pd.Series({
        'rev': wavg(d,'Revenue'), 'cog': wavg(d,'COGS')}))
    mn = s.groupby('M').apply(lambda d: pd.Series({
        'rev': wavg(d,'Revenue'), 'cog': wavg(d,'COGS')}))

    sales_dict = {pd.Timestamp(t): (r, c)
                  for t, r, c in zip(s['Date'].values, s['Revenue'].values, s['COGS'].values)}
    gr = float(s['Revenue'].mean()); gc = float(s['COGS'].mean())

    rows = []
    for d in pd.to_datetime(dates):
        k_md, k_dw = (d.month, d.day), (d.dayofweek, int(d.isocalendar().week))
        md_r = md.loc[k_md, 'rev'] if k_md in md.index else gr
        md_c = md.loc[k_md, 'cog'] if k_md in md.index else gc
        dw_r = dw.loc[k_dw, 'rev'] if k_dw in dw.index else gr
        dw_c = dw.loc[k_dw, 'cog'] if k_dw in dw.index else gc
        mn_r = mn.loc[d.month, 'rev']; mn_c = mn.loc[d.month, 'cog']

        l365 = sales_dict.get(d - pd.Timedelta(days=365), (md_r, md_c))
        l730 = sales_dict.get(d - pd.Timedelta(days=730), (md_r, md_c))
        l1095 = sales_dict.get(d - pd.Timedelta(days=1095), (md_r, md_c))

        rows.append({
            'enc_md_rev': md_r, 'enc_md_cog': md_c,
            'enc_dw_rev': dw_r, 'enc_dw_cog': dw_c,
            'enc_mn_rev': mn_r, 'enc_mn_cog': mn_c,
            'lag365_rev': l365[0], 'lag365_cog': l365[1],
            'lag730_rev': l730[0], 'lag730_cog': l730[1],
            'lag1095_rev': l1095[0], 'lag1095_cog': l1095[1],
        })
    return pd.DataFrame(rows, index=range(len(rows)))

print('Đang trích xuất đặc trưng quá khứ...')
lag_tr = build_lag_features(feat['Date'], sales)
lag_te = build_lag_features(test_df['Date'], sales)

# Chuyển sang logarit: target là log nên feature quá khứ ở cùng tỷ lệ sẽ ổn định hơn
for col in lag_tr.columns:
    lag_tr[col] = np.log(np.maximum(lag_tr[col].astype(float), 1.0))
    lag_te[col] = np.log(np.maximum(lag_te[col].astype(float), 1.0))

feat = pd.concat([feat.reset_index(drop=True), lag_tr.reset_index(drop=True)], axis=1)
test_df = pd.concat([test_df.reset_index(drop=True), lag_te.reset_index(drop=True)], axis=1)

NON_FEATURES = {'Date','Revenue','COGS'}
cols = [c for c in feat.columns if c not in NON_FEATURES]

X_tr = feat[cols].values.astype(float)
X_te = test_df[cols].values.astype(float)
y_rev = np.log(feat['Revenue'].values)
y_cog = np.log(feat['COGS'].values)
years = feat['Date'].dt.year.values

print(f'Tập Train: {X_tr.shape}, Tập Test: {X_te.shape}, Số lượng đặc trưng: {len(cols)}')


## 5. Trọng số mẫu (Sample Weights)


In [ ]:
# Trọng số mẫu: Tập trung vào giai đoạn ổn định 2014-2018 (Theo benchmark 658k)
w_full = np.full(len(years), 0.01)
w_full[(years >= 2014) & (years <= 2018)] = 1.0

print(f'Phân bổ trọng số:')
print(f'  Ngày có w=1.0: {(w_full == 1.0).sum()}')
print(f'  Ngày có w=0.01: {(w_full == 0.01).sum()}')


## 6. Mô hình 1 — Ridge Regression (xu hướng tuyến tính)


In [ ]:
def train_ridge(X_train, y_train, alpha=3.0):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0).replace(0, 1)
    Xs = (X_train - mu) / sigma
    m = Ridge(alpha=alpha, random_state=42)
    m.fit(Xs, y_train)
    return m, (mu, sigma)

def predict_ridge(model, X_test, stats):
    mu, sigma = stats
    return model.predict((X_test - mu) / sigma)

ridge_rev, st_r = train_ridge(pd.DataFrame(X_tr, columns=cols), y_rev)
ridge_cog, st_c = train_ridge(pd.DataFrame(X_tr, columns=cols), y_cog)

p_rd_rev = np.exp(predict_ridge(ridge_rev, pd.DataFrame(X_te, columns=cols), st_r))
p_rd_cog = np.exp(predict_ridge(ridge_cog, pd.DataFrame(X_te, columns=cols), st_c))

print(f'Ridge Doanh thu: {p_rd_rev.mean():,.0f}')
print(f'Ridge Chi phí:   {p_rd_cog.mean():,.0f}')


## 7. Mô hình 2 — LightGBM với Huber Loss


In [ ]:
LGB_PARAMS = dict(
    objective='regression', metric='mae',
    learning_rate=0.03, num_leaves=63,
    min_data_in_leaf=30,
    feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
    lambda_l2=1.0, seed=42, verbosity=-1,
)

def train_lgb(X, y, w, num_boost_es=5000, early_stop=300):
    intern = pd.Timestamp('2022-07-04')
    fit_idx = (feat['Date'] <= intern).values
    ins_idx = (feat['Date'] >  intern).values

    bk = lgb.train(
        LGB_PARAMS,
        lgb.Dataset(X[fit_idx], y[fit_idx], weight=w[fit_idx]),
        num_boost_round=num_boost_es,
        valid_sets=[lgb.Dataset(X[ins_idx], y[ins_idx])],
        callbacks=[lgb.early_stopping(early_stop, verbose=False),
                   lgb.log_evaluation(0)])

    bf = lgb.train(LGB_PARAMS,
                   lgb.Dataset(X, y, weight=w),
                   num_boost_round=bk.best_iteration)
    return bf, bk.best_iteration

print('Huấn luyện mô hình LGBM cơ sở...')
bf_rev, best_iter_rev = train_lgb(X_tr, y_rev, w_full)
p_lgb_rev = np.exp(bf_rev.predict(X_te))

bf_cog, best_iter_cog = train_lgb(X_tr, y_cog, w_full)
p_lgb_cog = np.exp(bf_cog.predict(X_te))


In [ ]:
# 7.5. Mô hình 2.5 — CatBoost (Thành phần kết hợp)
from catboost import CatBoostRegressor

def train_cb(X, y, w):
    params = dict(
        loss_function='RMSE',
        learning_rate=0.03,
        depth=8,
        iterations=2500,
        l2_leaf_reg=3.0,
        random_seed=42,
        verbose=False,
        thread_count=-1
    )
    model = CatBoostRegressor(**params)
    model.fit(X, y, sample_weight=w)
    return model

print('Đang huấn luyện CatBoost Doanh thu...')
cb_rev = train_cb(X_tr, y_rev, w_full)
p_cb_rev = np.exp(cb_rev.predict(X_te))

print('Đang huấn luyện CatBoost Chi phí...')
cb_cog = train_cb(X_tr, y_cog, w_full)
p_cb_cog = np.exp(cb_cog.predict(X_te))

print(f'CatBoost Doanh thu: {p_cb_rev.mean():,.0f}')
print(f'CatBoost Chi phí:   {p_cb_cog.mean():,.0f}')


## 8. Mô hình 3 — Prophet (chuỗi thời gian)


In [ ]:
def build_promo_regressors(dates):
    full = build_features(dates)
    promo_cols = [c for c in full.columns
                  if c.startswith('promo_') and c.count('_') == 1]
    return full[['Date'] + promo_cols].rename(columns={'Date':'ds'})

def fit_prophet(train_df, post_regime_only=True):
    if post_regime_only:
        train_df = train_df[train_df['ds'] >= '2020-01-01']
    m = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                daily_seasonality=False,
                seasonality_mode='multiplicative',
                changepoint_prior_scale=0.05)
    for col in [c for c in train_df.columns if c.startswith('promo_')]:
        m.add_regressor(col)
    m.fit(train_df)
    return m

print('Train Prophet Revenue...')
tdf_r = pd.DataFrame({'ds': sales['Date'], 'y': np.log(sales['Revenue'])}) \
          .merge(build_promo_regressors(sales['Date']), on='ds')
mp_r = fit_prophet(tdf_r)

print('Train Prophet COGS...')
tdf_c = pd.DataFrame({'ds': sales['Date'], 'y': np.log(sales['COGS'])}) \
          .merge(build_promo_regressors(sales['Date']), on='ds')
mp_c = fit_prophet(tdf_c)

vdf = pd.DataFrame({'ds': test_df['Date']}) \
        .merge(build_promo_regressors(test_df['Date']), on='ds')
p_pr_rev = np.exp(mp_r.predict(vdf)['yhat'].values)
p_pr_cog = np.exp(mp_c.predict(vdf)['yhat'].values)

print(f'Prophet Revenue: {p_pr_rev.mean():,.0f}')
print(f'Prophet COGS:    {p_pr_cog.mean():,.0f}')


## 9. Q-Specialist (8 mô hình LightGBM)


In [ ]:
def train_q_specialist(X, y, w_base, target_q, q_boost=2.0):
    Q_train = feat['Date'].dt.quarter.values
    w = w_base.copy()
    w[Q_train == target_q] *= q_boost
    # Gọi train_lgb đúng tham số đã định nghĩa
    bf, _ = train_lgb(X, y, w, num_boost_es=3000, early_stop=200)
    return bf

spec_rev = {}
spec_cog = {}
for q in [1, 2, 3, 4]:
    print(f'Mô hình chuyên biệt Q{q} Doanh thu...')
    bf = train_q_specialist(X_tr, y_rev, w_full, q)
    spec_rev[q] = np.exp(bf.predict(X_te))

    print(f'Mô hình chuyên biệt Q{q} Chi phí...')
    bf = train_q_specialist(X_tr, y_cog, w_full, q)
    spec_cog[q] = np.exp(bf.predict(X_te))

# Ghép nối kết quả
Q_test = test_df['Date'].dt.quarter.values
lgb_spec_rev = np.zeros(len(test_df))
lgb_spec_cog = np.zeros(len(test_df))
for q in [1,2,3,4]:
    mask = Q_test == q
    lgb_spec_rev[mask] = spec_rev[q][mask]
    lgb_spec_cog[mask] = spec_cog[q][mask]

print('Hoàn thành huấn luyện 8 mô hình Specialist.')


## 10. Tổng hợp & Xuất kết quả dự báo cuối cùng


In [ ]:
required = [
    'sales', 'test_df', 'OUT_DIR',
    'lgb_spec_rev', 'lgb_spec_cog',
    'p_lgb_rev', 'p_lgb_cog',
    'p_cb_rev', 'p_cb_cog',
    'p_pr_rev', 'p_pr_cog',
    'p_rd_rev', 'p_rd_cog',
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError(f'Vui lòng chạy tất cả các cell mô hình phía trên (bao gồm CatBoost), đang thiếu: {missing}')

CFG = {
    'cr': {1: 1.22, 2: 1.24, 3: 1.34, 4: 1.32},
    'cc': {1: 1.28, 2: 1.30, 3: 1.40, 4: 1.36},
    'l4': {'alpha': 0.65, 'w': (0.611111, 0.277778, 0.055556, 0.055556, 0.0)},
    'p3': {'alpha': 0.72, 'w': (0.511364, 0.397727, 0.045455, 0.045455, 0.0)},
    'mix_r4': 0.70,
    'mix_t2': (0.80, 0.15, 0.05),
    'mix_t4': (0.60, 0.40),
}

def build_base_submission():
    q = test_df['Date'].dt.quarter.values
    date_str = test_df['Date'].dt.strftime('%Y-%m-%d').values
    
    # Year Growth Calibration (Hệ số đưa raw prediction lên level 2023-2024)
    cr = np.array([CFG['cr'][int(x)] for x in q], dtype=float)
    cc = np.array([CFG['cc'][int(x)] for x in q], dtype=float)

    def _layer(alpha, w):
        w_lgb, w_cb, w_pr, w_rd, w_nv = w
        s = sum(w) if sum(w) > 0 else 1.0
        w_lgb, w_cb, w_pr, w_rd, w_nv = [x/s for x in w]
        
        br = alpha * lgb_spec_rev + (1 - alpha) * p_lgb_rev
        bc = alpha * lgb_spec_cog + (1 - alpha) * p_lgb_cog
        
        # Naive predictions (zeros for n00 config)
        nv_r = nv_c = np.zeros(len(test_df))
        
        raw_r = w_lgb * br + w_cb * p_cb_rev + w_pr * p_pr_rev + w_rd * p_rd_rev + w_nv * nv_r
        raw_c = w_lgb * bc + w_cb * p_cb_cog + w_pr * p_pr_cog + w_rd * p_rd_cog + w_nv * nv_c
        
        return np.maximum(cr * raw_r, 0.0), np.maximum(cc * raw_c, 0.0)

    l4_r, l4_c = _layer(CFG['l4']['alpha'], CFG['l4']['w'])
    p3_r, p3_c = _layer(CFG['p3']['alpha'], CFG['p3']['w'])
    
    r4_r = CFG['mix_r4'] * l4_r + (1 - CFG['mix_r4']) * p3_r
    r4_c = CFG['mix_r4'] * l4_c + (1 - CFG['mix_r4']) * p3_c

    wa, wb, wc = CFG['mix_t2']
    t2_r = wa * r4_r + wb * l4_r + wc * p3_r
    t2_c = wa * r4_c + wb * l4_c + wc * p3_c

    t3_r = np.median(np.vstack([r4_r, l4_r, p3_r]), axis=0)
    t3_c = np.median(np.vstack([r4_c, l4_c, p3_c]), axis=0)

    w2, w3 = CFG['mix_t4']
    out_r = w2 * t2_r + w3 * t3_r
    out_c = w2 * t2_c + w3 * t3_c

    return pd.DataFrame({'Date': date_str, 'Revenue': out_r, 'COGS': out_c})

best = build_base_submission()
out_path = OUT_DIR / 'submission.csv'
best.to_csv(out_path, index=False)
print('Đã lưu kết quả tại:', out_path)
best.head()

In [ ]:
FEATURE_MAPPING = {
    'enc_md_rev': 'Lịch sử Doanh thu (Cùng kỳ)',
    'enc_md_cog': 'Lịch sử Chi phí (Cùng kỳ)',
    'enc_dw_rev': 'Lịch sử Doanh thu (Theo Thứ/Tuần)',
    'enc_dw_cog': 'Lịch sử Chi phí (Theo Thứ/Tuần)',
    'lag365_rev': 'Doanh thu Trễ (365 Ngày)',
    'lag365_cog': 'Chi phí Trễ (365 Ngày)',
    'lag730_rev': 'Doanh thu Trễ (730 Ngày)',
    'lag730_cog': 'Chi phí Trễ (730 Ngày)',
    'lag1095_rev': 'Doanh thu Trễ (1095 Ngày)',
    'lag1095_cog': 'Chi phí Trễ (1095 Ngày)',
    't_days': 'Xu hướng Biến đổi (Tính bằng Ngày)',
    't_years': 'Xu hướng Biến đổi (Tính bằng Năm)',
    'year': 'Năm Hệ thống',
    'dow': 'Thứ trong tuần (DOW)',
    'regime_2019': 'Chế độ Tiền Khủng hoảng (2019)',
    'regime_pre2019': 'Chế độ Tiền Khủng hoảng (<2019)',
    'promo_urban_blowout_since': 'Khoảng cách từ đợt Urban Blowout',
    'promo_urban_blowout_until': 'Thời gian đếm ngược Urban Blowout',
    'tet_days_diff': 'Số ngày chênh lệch với Tết Âm lịch',
    'cos_m1': 'Biến đổi Mùa vụ (Fourier Cos M1)',
    'cos_m2': 'Biến đổi Mùa vụ (Fourier Cos M2)',
    'sin_m2': 'Biến đổi Mùa vụ (Fourier Sin M2)',
    'sin_w1': 'Biến đổi Mùa vụ (Fourier Sin W1)',
}

import shap
import matplotlib.pyplot as plt
import numpy as np

# ---- Chỉ sử dụng SHAP Analysis theo yêu cầu ----
print("Đang tính toán SHAP values cho Mô hình Doanh thu (lấy mẫu 500 điểm)...")
try:
    explainer = shap.TreeExplainer(bf_rev)
    # Lấy mẫu ngẫu nhiên để đảm bảo tốc độ xử lý
    np.random.seed(42)
    sample_idx = np.random.choice(X_tr.shape[0], 500, replace=False)
    X_sample = X_tr[sample_idx]
    shap_values = explainer.shap_values(X_sample)

    # Dịch tên biến sang tiếng Việt dựa trên FEATURE_MAPPING
    display_cols = [FEATURE_MAPPING.get(c, c.replace('_', ' ').title()) for c in cols]

    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_sample, feature_names=display_cols, show=False)
    plt.title("Phân tích SHAP: Tác động của các đặc trưng tới Doanh thu", fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig("shap_analysis.png", dpi=300, bbox_inches='tight')
    plt.show()
    print("Đã lưu ảnh shap_analysis.png")
except Exception as e:
    print(f"Lỗi khi chạy SHAP: {e}. Vui lòng đảm bảo đã cài đặt thư viện 'shap'.")
